# LLMBackend Usage

`LLMBackend` implements krrood's `GenerativeBackend`: when a `Match` contains `...`, the backend fills those free slots from the LLM and the serialized `SymbolGraph` world context.

Current usage patterns:
1. **`resolve_params()`** — action type is known; return a concrete action instance without creating a plan node.
2. **`resolve_match()`** — action type is known; return a PyCRAM plan node for the resolved match.
3. **Context-level `LLMBackend`** — install the backend on a context and use `execute_single()`.
4. **Mid-session swap** — mix a normal krrood backend and `LLMBackend` in the same session.


In [ ]:
from uniworld import load_pr2_apartment_world
from semantic_digital_twin.world_description.world_entity import Body

world, pr2, context = load_pr2_apartment_world()
groundable_type = Body  # optional — defaults to Symbol inside llmr


In [ ]:
from dotenv import load_dotenv
load_dotenv("../.env")
from llmr.reasoning.llm_config import make_llm, LLMProvider

llm = make_llm(LLMProvider.OPENAI, model="gpt-4o-mini")

---
## 1 — Known Action Type: `resolve_params()` and `resolve_match()`

Use `resolve_params()` when you only want the resolved action instance. Use `resolve_match()` when you want a `PlanNode` that can later be performed.


In [ ]:
from krrood.entity_query_language.query.match import Match
from pycram.robot_plans.actions.core.pick_up import PickUpAction

from llmr import resolve_match, resolve_params

INSTRUCTION = "pick up the milk from the table"

param_match = Match(PickUpAction)(
    object_designator=...,
    arm=...,
    grasp_description=...,
)

action_instance = resolve_params(
    match=param_match,
    llm=llm,
    groundable_type=groundable_type,
    instruction=INSTRUCTION,
    strict_required=True,
)

print("Resolved action type:", type(action_instance).__name__)
print(action_instance)


In [ ]:
plan_match = Match(PickUpAction)(
    object_designator=...,
    arm=...,
    grasp_description=...,
)

plan_node = resolve_match(
    match=plan_match,
    context=context,
    llm=llm,
    groundable_type=groundable_type,
    instruction=INSTRUCTION,
    strict_required=True,
)

print("Backend type:", type(context.query_backend).__name__)
print("Plan node   :", plan_node)


In [ ]:
action_instance


In [ ]:
plan_node.__dict__


In [ ]:
# Run only in a live simulation environment.
# plan_node.perform()


In [ ]:
# Inspect the generated plan graph after execution, if available.
# plan_node.plan.plan_graph.nodes()


---
## 2 — Context-level

Pass `LLMBackend` at context construction. Every plan in this context goes
through the LLM — no per-plan wiring needed.  Update `instruction` per-plan.


In [ ]:
from krrood.entity_query_language.query.match import Match
from pycram.robot_plans.actions.core.pick_up import PickUpAction
from pycram.robot_plans.actions.core.navigation import NavigateAction
from pycram.datastructures.dataclasses import Context

from llmr.backend import LLMBackend
from llmr.pycram_bridge import execute_single

llm_context = Context.from_world(
    world,
    query_backend=LLMBackend(
        llm=llm,
        groundable_type=groundable_type,
        instruction="go to the kitchen table",
        strict_required=True,
    ),
)

print("Backend type:", type(llm_context.query_backend).__name__)


In [ ]:
# Plan 1 — uses the backend set on llm_context.
nav_match = Match(NavigateAction)(target_location=...)
nav_node = execute_single(nav_match, llm_context)

# Run only in a live simulation environment.
# nav_node.perform()


In [ ]:
# Plan 2 — update instruction for the next action, same context.
llm_context.query_backend.instruction = "pick up the milk from the table"

pick_match = Match(PickUpAction)(object_designator=..., arm=..., grasp_description=...)
pick_node = execute_single(pick_match, llm_context)

# Run only in a live simulation environment.
# pick_node.perform()


---
## 3 — Mid-session swap

Start with `ProbabilisticBackend` for routine actions. Switch to `LLMBackend`
via `resolve_match()` for novel underspecified actions, then restore.


In [ ]:
from krrood.entity_query_language.backends import ProbabilisticBackend
from krrood.entity_query_language.query.match import Match
from pycram.robot_plans.actions.core import PickUpAction, PlaceAction

from llmr.pycram_bridge import execute_single
from llmr.backend import LLMBackend

# Session starts with probabilistic backend.
prob_backend = ProbabilisticBackend()
context.query_backend = prob_backend
print("Start backend:", type(context.query_backend).__name__)


In [ ]:
# Routine pick — handled by the currently installed backend.
pick_match = Match(PickUpAction)(object_designator=..., arm=..., grasp_description=...)
pick_node = execute_single(pick_match, context)

# Run only in a live simulation environment.
# pick_node.perform()


In [ ]:
from krrood.entity_query_language.query.match import Match
from pycram.robot_plans.actions.core import PlaceAction

from llmr import resolve_match

# Novel action — use resolve_match() for this one underspecified action.
place_match = Match(PlaceAction)(object_designator=..., target_location=...)

place_node = resolve_match(
    match=place_match,
    context=context,
    llm=llm,
    groundable_type=groundable_type,
    instruction="put the milk gently inside the fridge on the top shelf",
    strict_required=True,
)
print("Swapped backend:", type(context.query_backend).__name__)

# Run only in a live simulation environment.
# place_node.perform()

context.query_backend = prob_backend
print("Restored backend:", type(context.query_backend).__name__)


In [ ]:
# Restore probabilistic backend for remaining session.
context.query_backend = prob_backend
print("Restored backend:", type(context.query_backend).__name__)
